# Hamiltonian evolution with QSVT

Regenerate the cosine and sine phases, compose their quantum sequences in native
qfunc, and compare the complete state with an independent sequential reference.
The original six-qubit Circuit packs each response into a dense matrix; keep it
as a separately measured reference, not a decomposed gate-count competitor.

The native route uses named parity, branch, selection and data ports, four-term
Pauli PREPARE/SELECT, controlled helpers and adjoints. It retains all 33 cosine
and 34 sine phases, time 22, scale 0.9999 and the 1e-7 evolution target. It does
not classically assemble the QSVT response matrix. Current lowering needs ten
qubits including four clean work bits, with an explicit 3500-operation budget.
The original Circuit route uses six qubits and opaque matrix instructions.

Try another input seed while retaining the Hamiltonian and phases. Complete
complex amplitudes must match without global-phase alignment; response flags
need not be clean. Work bits must return to zero. Read postselection probability
separately from the rescaled vector error and exact probabilities separately
from finite-shot measurements. Phase fitting, execution and gate efficiency are
three independent claims.


Use the local FlagQuantum environment.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Define the four-term Hamiltonian, evolution time and tolerance.

In [ ]:
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview import (
    workflow,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview.main import (
    print_report,
    save_figures,
)

print(fq.__name__)
print(workflow.WORDS, workflow.WEIGHTS, workflow.TIME, workflow.EPS)


Construct the checked Pauli LCU block from its four terms.

In [ ]:
block = workflow.encoding()
print(block.target, block.alpha)


Generate one reproducible real input and normalize it.

In [ ]:
amplitudes = workflow.input_state()
print(amplitudes)


Inspect the two extra flags for extracting and combining the QSVT responses.

In [ ]:
print("Data, Hamiltonian block, response and LCU widths:", 2, 2, 1, 1)


Build the Jacobi–Anger polynomials at their original degree and scale.

In [ ]:
print("Series degree and scaling:", workflow.DEGREE, workflow.SCALE)


Apply every phase and alternating block/adjoint through named quantum ports.
The classical builder binds fitted angles; the visible qfunc owns the sequence.


In [ ]:
result = workflow.run()
print_report(result)
from flagquantum.synth import (
    H,
    Output,
    QArray,
    QBit,
    QNum,
    adjoint,
    allocate,
    control,
    phase,
    prepare_amplitudes,
    qfunc,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview import (
    quantum,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview.quantum import (
    mixed_rotation,
    projector_rotation,
    signal,
)


@qfunc
def main(
    block_weights: list[float],
    even: list[float],
    odd: list[float],
    weights: list[float],
    amplitudes: list[float],
    parity: Output[QBit],
    branch: Output[QBit],
    selection: Output[QNum[2]],
    data: Output[QArray[QBit]],
) -> None:
    allocate(parity)
    allocate(branch)
    allocate(selection)
    prepare_amplitudes(amplitudes, 0, data)
    H(parity)
    H(branch)
    mixed_rotation(even[0], odd[0], parity, branch, selection)
    for pair in range(16):
        with adjoint():
            signal(block_weights, selection, data)
        mixed_rotation(even[2 * pair + 1], odd[2 * pair + 1], parity, branch, selection)
        signal(block_weights, selection, data)
        mixed_rotation(even[2 * pair + 2], odd[2 * pair + 2], parity, branch, selection)
    with adjoint():
        control(parity, lambda: signal(block_weights, selection, data))
    control(parity, lambda: projector_rotation(odd[33], branch, selection))
    phase(1, weights[0])
    phase(branch, weights[1] - weights[0])
    phase(parity, weights[2] - weights[0])
    phase(branch * parity, weights[3] - weights[2] - weights[1] + weights[0])
    H(branch)
    H(parity)


# Execute this editable definition through the example's bounded builder.
quantum.main = main
program = quantum.build_program(result.cosine, result.sine, amplitudes)
executed = fq.run(program)
print(program.registers, program.report["resources"])


Verify the Hamiltonian block projection.

In [ ]:
assert result.block_error < 1e-9


Check the alternating even and odd phase streams.

In [ ]:
assert abs(result.cosine.phases.numel() - result.sine.phases.numel()) <= 1


Read the complete state comparison and the represented circuit width.

In [ ]:
print("Full complex-state error:", result.full_state_error)
print("Circuit width:", result.resources.n_qubits)
import torch
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.qls_chebyshev_lcu.overview.encoding import (
    unitary,
)

# Output ports occupy six high bits; four low work bits must be clean.
full = executed.statevector().reshape(64, 16)
expected_full = workflow.full_reference(
    unitary(block), result.cosine, result.sine, amplitudes
)
torch.testing.assert_close(full[:, 0], expected_full, atol=1e-10, rtol=0)
assert float(full[:, 1:].abs().square().sum()) < 1e-24
print(
    "Native absolute complex-state error:",
    float((full[:, 0] - expected_full).abs().max()),
)
print(
    "Named selector-zero probability:",
    sum(
        probability
        for (parity, branch, selection), probability in program.probabilities(
            executed, registers=("parity", "branch", "selection")
        ).items()
        if parity == branch == selection == 0
    ),
)


Inspect the zero-block projection and rescale by the original response factor.
Compare exact success probability with one correlated finite-shot request.
Conditioning changes the denominator to accepted shots. This separate measured
Circuit plan is not the sealed Program execution and does not alter its state.


In [ ]:
print(
    "Projection probability and rescale factor:",
    result.success_probability,
    2 / workflow.SCALE,
)

# One joint sample request preserves correlation between flags and data.
# Selecting each port in a separate sample request would draw different shots.
read_ports = ("parity", "branch", "selection", "data")
sampled = fq.run(
    program.circuit,
    outputs=fq.samples(
        qubits=tuple(wire for name in read_ports for wire in program.registers[name]),
        name="joint",
    ),
    options=fq.ExecutionOptions(shots=4096, seed=37),
)
rows = sampled.measurement("joint").value[0]
flag_width = sum(len(program.registers[name]) for name in read_ports[:-1])
accepted = rows[:, :flag_width].sum(dim=1) == 0
accepted_shots = int(accepted.sum())
if accepted_shots == 0:
    raise ValueError(
        "No accepted shots; increase the declared shot budget before estimating a conditional mean"
    )
print("Sampled zero-flag probability:", accepted_shots / len(rows))
# This conditional estimate divides by accepted shots, not total shots.
print(
    "Conditional data bit-one frequencies:",
    rows[accepted, flag_width:].double().mean(dim=0),
)
print("Accepted / total shots:", accepted_shots, len(rows))


Compare against the matrix exponential and display both plots.

In [ ]:
assert result.evolution_error < workflow.EPS
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "qsvt-response-errors.png")))
display(Image(filename=str(figures / "qsvt-evolution-state.png")))
